# 01 · Electronic descriptors for aryl bromides

**4th-year project: ML-guided Suzuki–Miyaura couplings of para-substituted bromobenzenes**

This is the first of three notebooks:

What the notebooks cover:
-  **01 Electronic descriptors**: Run xTB and DFT calculations and extract descriptors (HOMO/LUMO, charges, dipoles, Fukui indices, ...) for the project substrates 
- **02 Suzuki yield prediction**: Train and *honestly evaluate* yield-prediction models on a published high-throughput Suzuki dataset usisng an existing deep-learning model (Chemprop) 
- **03 Bayesian optimisation** Use BO to choose which reactions to run next, first on a "virtual lab" and then on your own experiments 

```
SMILES ──► 3D structure ──► quantum chemistry ──► descriptors ──► yield model / Bayesian optimisation
            (RDKit)          (xTB, DFT)           
```

**By the end of this notebook you should be able to:**
1. Turn a SMILES string into a sensible 3D geometry, and explain why conformers matter.
2. Run semi-empirical (GFN2-xTB) and DFT (B3LYP/def2-SVP) calculations from Python.
3. Extract electronic descriptors: frontier orbital energies, atomic partial charges, dipole moments, conceptual-DFT quantities (electronegativity, hardness, electrophilicity, Fukui indices), and visualise orbitals and electrostatic potentials.
4. Check that your descriptors make chemical sense by comparing them with Hammett σ constants, and know their limitations.
5. Export a descriptor table for the substrates.

**How to use this notebook:** Run the cells in order. Cells marked **Your turn** are exercises. Don't skip them: they are where most of the learning happens. Keep a lab-book noting what you ran, what you found, and anything that surprised you.

**Further reading:**
- Bannwarth, Ehlert, Grimme, *J. Chem. Theory Comput.* **2019**, 15, 1652: GFN2-xTB.
- Geerlings, De Proft, Langenaeker, *Chem. Rev.* **2003**, 103, 1793: conceptual DFT (hardness, Fukui functions, ...).
- Hansch, Leo, Taft, *Chem. Rev.* **1991**, 91, 165: Hammett substituent constants.
- Ahneman *et al.*, *Science* **2018**, 360, 186: DFT descriptors + random forests for C–N coupling yields (the classic descriptor-based yield paper).
- Żurański, Martinez Alvarado, Shields, Doyle, *Acc. Chem. Res.* **2021**, 54, 1856: review of reaction descriptors and ML for reaction yields.

## 0. Loading evdrything you need

In [ ]:
import os, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, Crippen, rdMolDescriptors, Draw
RDLogger.DisableLog("rdApp.*")

from ase import Atoms
from ase.optimize import BFGS
from tblite.ase import TBLite
from tblite.interface import Calculator as XTBCalculator

from pyscf import gto, dft, lib
from pyscf.tools import cubegen

import py3Dmol

lib.num_threads(os.cpu_count())      # let PySCF use all cores
DATA = Path("data"); DATA.mkdir(exist_ok=True)
RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)

# Unit conversions: quantum-chemistry codes work in atomic units internally
HARTREE2EV = 27.211386245988
BOHR2ANG = 0.529177210903
AU2DEBYE = 2.541746473

## 1. The project substrates: para-substituted bromobenzenes

In a Suzuki–Miyaura coupling, the aryl bromide undergoes **oxidative addition** to Pd(0). That step is often rate-limiting for aryl bromides and chlorides. It is faster for electron-poor arenes. The para substituent tunes the electron density at the C–Br carbon, and the Hammett σ<sub>p</sub> constant summarises that effect (positive = electron-withdrawing).

Below is an example of para substituted benzenes that might be useful to get an idea:
1. **A sanity check.** We know how the substituents should behave, so we can check that our descriptors capture it.
2. **The project substrates.** Their descriptors become inputs for the yield model and the Bayesian optimisation you will have to add your project related data below

Feel free to extend the list with any compounds **you will actually work with**. Look up σ<sub>p</sub> in Hansch *et al.* Notebook 03 picks up whatever you put here.

> **Two coupling partners.** Your reaction matrix varies both the aryl bromide *and* the arylboronic acid, so you need descriptors for both. The code below finds the reactive site with the SMARTS pattern `"c-Br"` (see `ipso_and_halogen`), which matches nothing in a boronic acid. Adapt it (e.g. `"c-B"`) and decide which atoms are the relevant sites for the boronic acid.

In [ ]:
hammett = pd.DataFrame([
    # name                       substituent  SMILES                      sigma_p (Hansch, Leo, Taft, Chem. Rev. 1991, 91, 165)
    ("4-aminobromobenzene",     "NH2",       "Nc1ccc(Br)cc1",            -0.66),
    ("4-bromoanisole",          "OMe",       "COc1ccc(Br)cc1",           -0.27),
    ("4-bromotoluene",          "Me",        "Cc1ccc(Br)cc1",            -0.17),
    ("bromobenzene",            "H",         "Brc1ccccc1",               +0.00),
    ("4-fluorobromobenzene",    "F",         "Fc1ccc(Br)cc1",            +0.06),
    ("4-chlorobromobenzene",    "Cl",        "Clc1ccc(Br)cc1",           +0.23),
    ("4-bromobenzotrifluoride", "CF3",       "FC(F)(F)c1ccc(Br)cc1",     +0.54),
    ("4-bromobenzonitrile",     "CN",        "N#Cc1ccc(Br)cc1",          +0.66),
    ("4-bromonitrobenzene",     "NO2",       "O=[N+]([O-])c1ccc(Br)cc1", +0.78),
], columns=["name", "substituent", "smiles", "sigma_p"])

Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in hammett.smiles],
                     legends=[f"{n}\nσp = {s:+.2f}" for n, s in zip(hammett.name, hammett.sigma_p)],
                     molsPerRow=5, subImgSize=(200, 160))

## 2. From SMILES to 3D structure

A SMILES string has no coordinates, but quantum chemistry needs them. Standard recipe:

1. Add hydrogens.
2. Generate several conformers with a distance-geometry method (RDKit's **ETKDG**).
3. Minimise each with a force field (**MMFF94**) and keep the lowest-energy one.
4. (Better) Re-optimise that geometry with a quantum method (xTB below, or DFT).

**Why several conformers?** Many descriptors depend on geometry: dipole moments very strongly, orbital energies less so. For flexible molecules (e.g. the phosphine ligands used in Suzuki couplings) you would normally do a proper conformer search (e.g. with [CREST](https://crest-lab.github.io/crest-docs/)). You would then either use the lowest conformer or Boltzmann-average the descriptors. For our rigid aryl halides, a quick search is enough.

In [ ]:
def embed_lowest_conformer(smiles, n_confs=20, seed=0xF00D):
    # SMILES -> RDKit Mol (with H) holding only its lowest-energy force-field conformer.
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    params = AllChem.ETKDGv3()
    params.randomSeed = seed
    cids = list(AllChem.EmbedMultipleConfs(mol, numConfs=n_confs, params=params))
    if not cids:
        raise RuntimeError(f"Embedding failed for {smiles}")
    if AllChem.MMFFHasAllMoleculeParams(mol):
        res = AllChem.MMFFOptimizeMoleculeConfs(mol, maxIters=2000)
    else:  # fall back to UFF for unusual elements
        res = AllChem.UFFOptimizeMoleculeConfs(mol, maxIters=2000)
    energies = [e for _, e in res]
    best = cids[int(np.argmin(energies))]
    out = Chem.Mol(mol)
    out.RemoveAllConformers()
    out.AddConformer(Chem.Conformer(mol.GetConformer(best)), assignId=True)
    return out


def show3d(mol, width=350, height=280):
    view = py3Dmol.view(width=width, height=height)
    view.addModel(Chem.MolToMolBlock(mol), "mol")
    view.setStyle({"stick": {}})
    view.zoomTo()
    return view.show()


mols = {row.name: embed_lowest_conformer(row.smiles) for row in hammett.itertuples()}
show3d(mols["4-bromoanisole"])

### 2b. Refine the geometry with GFN2-xTB

[GFN2-xTB](https://xtb-docs.readthedocs.io) is a *semi-empirical* tight-binding method. It is roughly 1000× cheaper than DFT and gives reasonable geometries for most organic and organometallic molecules. We drive it through the [ASE](https://wiki.fysik.dtu.dk/ase/) optimiser using the `tblite` library.

A common, cost-effective protocol is **xTB geometry → DFT single-point energy/properties**.

In [ ]:
def rdkit_to_ase(mol):
    return Atoms(numbers=[a.GetAtomicNum() for a in mol.GetAtoms()],
                 positions=mol.GetConformer().GetPositions())


def xtb_optimise(mol, charge=0, fmax=0.03, steps=500):
    # Return a copy of `mol` with GFN2-xTB-optimised coordinates (Angstrom).
    atoms = rdkit_to_ase(mol)
    atoms.calc = TBLite(method="GFN2-xTB", charge=charge, verbosity=0)
    BFGS(atoms, logfile=None).run(fmax=fmax, steps=steps)
    new = Chem.Mol(mol)
    conf = new.GetConformer()
    for i, p in enumerate(atoms.get_positions()):
        conf.SetAtomPosition(i, p.tolist())
    return new


t0 = time.time()
mols_xtb = {name: xtb_optimise(m) for name, m in mols.items()}
print(f"xTB optimisations: {time.time() - t0:.1f} s for {len(mols_xtb)} molecules")

# Sanity check: does the C-Br bond length respond to the substituent?
def ipso_and_halogen(mol, smarts="c-Br"):
    # Indices of (ipso carbon, halogen). Be specific: "c-[Cl,Br,I]" would pick the
    # C-Cl bond in 4-chlorobromobenzene. Always check which atoms a SMARTS matches!
    return mol.GetSubstructMatch(Chem.MolFromSmarts(smarts))

for name, m in mols_xtb.items():
    c, x = ipso_and_halogen(m)
    d = AllChem.GetBondLength(m.GetConformer(), c, x)
    print(f"{name:26s} C-Br = {d:.3f} Å")

## 3. Descriptor level 0: cheap 2D descriptors (RDKit)

Start with "free" descriptors that need no 3D structure. They are a useful **baseline**. If a fancy quantum descriptor does no better than molecular weight, that tells you something!

In [ ]:
def rdkit_descriptors(mol):
    m = Chem.RemoveHs(mol)
    AllChem.ComputeGasteigerCharges(m)
    c, x = ipso_and_halogen(m)
    return {
        "MW": Descriptors.MolWt(m),
        "logP": Crippen.MolLogP(m),
        "TPSA": rdMolDescriptors.CalcTPSA(m),
        "q_gasteiger_ipso": m.GetAtomWithIdx(c).GetDoubleProp("_GasteigerCharge"),
    }

desc_rdkit = pd.DataFrame({n: rdkit_descriptors(m) for n, m in mols.items()}).T
desc_rdkit

## 4. Descriptor level 1: electronic descriptors from GFN2-xTB

From a single xTB calculation we can read off:

| Descriptor | Meaning | Chemical relevance |
|---|---|---|
| $\varepsilon_\mathrm{HOMO}$ | energy of highest occupied MO | nucleophilicity, ease of oxidation |
| $\varepsilon_\mathrm{LUMO}$ | energy of lowest unoccupied MO | electrophilicity, ease of reduction |
| gap | $\varepsilon_\mathrm{LUMO}-\varepsilon_\mathrm{HOMO}$ | kinetic stability / "hardness" |
| atomic charges $q_A$ | Mulliken-type charges | local electron density at reactive sites |
| dipole $\mu$ | molecular polarity | solvation, intermolecular interactions |

### Conceptual DFT

By adding or removing one electron (a *vertical* calculation at the same geometry) we get the ionisation potential and electron affinity:

$$IP = E(N-1) - E(N), \qquad EA = E(N) - E(N+1)$$

and from them the **global reactivity indices**:

$$\chi = \tfrac{1}{2}(IP + EA)\ \text{(electronegativity)}, \quad \eta = IP - EA\ \text{(hardness)}, \quad \omega = \frac{\chi^2}{2\eta}\ \text{(electrophilicity)}$$

The **condensed Fukui functions** say *where* on the molecule electrons are added or removed:

$$f_A^+ = q_A(N) - q_A(N+1)\ \text{(site for nucleophilic attack)}, \qquad f_A^- = q_A(N-1) - q_A(N)\ \text{(site for electrophilic attack)}$$

(Conventions for η differ by a factor of 2 between papers. Be consistent and say which one you use.)

In [ ]:
def xtb_singlepoint(mol, charge=0, uhf=0):
    numbers = np.array([a.GetAtomicNum() for a in mol.GetAtoms()])
    pos_bohr = mol.GetConformer().GetPositions() / BOHR2ANG
    calc = XTBCalculator("GFN2-xTB", numbers, pos_bohr, charge=charge, uhf=uhf)
    calc.set("verbosity", 0)
    return calc.singlepoint()


def frontier_orbitals(eps, occ):
    # HOMO/LUMO energies from orbital energies and occupations (any units).
    occupied = np.where(occ > 0.5)[0]   # occupations are ~2 (restricted) or ~1 (unrestricted)
    homo = occupied.max()
    return eps[homo], eps[homo + 1]


def conceptual_dft(E_N, E_cat, E_an, q_N, q_cat, q_an):
    # Global + condensed (per-atom) conceptual-DFT descriptors. Energies in eV.
    IP, EA = E_cat - E_N, E_N - E_an
    chi = 0.5 * (IP + EA)
    eta = IP - EA
    return {
        "IP": IP, "EA": EA, "chi": chi, "eta": eta, "omega": chi**2 / (2 * eta),
        "f_plus": q_N - q_an,     # arrays: one value per atom
        "f_minus": q_cat - q_N,
    }


def xtb_descriptors(mol, site_atoms=None):
    # Electronic descriptors from GFN2-xTB. `site_atoms` = {label: atom index} for atom-specific ones.
    res = xtb_singlepoint(mol)
    cat = xtb_singlepoint(mol, charge=+1, uhf=1)
    an = xtb_singlepoint(mol, charge=-1, uhf=1)

    homo, lumo = frontier_orbitals(res.get("orbital-energies") * HARTREE2EV,
                                   res.get("orbital-occupations"))
    q = res.get("charges")
    cdft = conceptual_dft(res.get("energy") * HARTREE2EV, cat.get("energy") * HARTREE2EV,
                          an.get("energy") * HARTREE2EV, q, cat.get("charges"), an.get("charges"))
    d = {
        "HOMO": homo, "LUMO": lumo, "gap": lumo - homo,
        "dipole": np.linalg.norm(res.get("dipole")) * AU2DEBYE,
        **{k: cdft[k] for k in ["IP", "EA", "chi", "eta", "omega"]},
    }
    for label, idx in (site_atoms or {}).items():
        d[f"q_{label}"] = q[idx]
        d[f"f+_{label}"] = cdft["f_plus"][idx]
        d[f"f-_{label}"] = cdft["f_minus"][idx]
    return d


def aryl_halide_sites(mol):
    c, x = ipso_and_halogen(mol)
    return {"ipso": c, "X": x}


t0 = time.time()
desc_xtb = pd.DataFrame({n: xtb_descriptors(m, aryl_halide_sites(m)) for n, m in mols_xtb.items()}).T
print(f"{time.time() - t0:.1f} s")
desc_xtb.round(3)

> **A word of caution on xTB orbital energies.** GFN2-xTB is parametrised for geometries and energies, not orbital energies. Its absolute HOMO/LUMO values are not physically meaningful (note how deep the LUMO is), but *trends within a series* often are. The IP/EA from energy differences (ΔSCF) are more reliable. You can check this against DFT. 

## 5. Descriptor level 2: DFT with PySCF

Now the "real" thing. [PySCF](https://pyscf.org) is a Python quantum-chemistry package. It is pip/conda-installable and fine for molecules up to ~50 atoms on a laptop. For bigger systems or production runs, use ORCA or Gaussian on the cluster.

**Level of theory:** B3LYP/def2-SVP. This is a cheap, standard choice for descriptors, not a recommendation for accurate energetics. `def2` basis sets use effective core potentials (ECPs) for elements heavier than Kr (e.g. iodine), so `to_pyscf` adds `ecp=basis` when the molecule contains such elements.

In [ ]:
def to_pyscf(mol, basis="def2-svp", charge=0, spin=0):
    atoms = [(a.GetSymbol(), tuple(p)) for a, p in zip(mol.GetAtoms(), mol.GetConformer().GetPositions())]
    heavy = any(a.GetAtomicNum() > 36 for a in mol.GetAtoms())   # beyond Kr -> needs an ECP
    return gto.M(atom=atoms, basis=basis, ecp=basis if heavy else None, charge=charge, spin=spin,
                 unit="Angstrom", verbose=0)


def run_dft(mol, xc="b3lyp", basis="def2-svp", charge=0, spin=0):
    pmol = to_pyscf(mol, basis, charge, spin)
    mf = (dft.RKS(pmol) if spin == 0 else dft.UKS(pmol)).density_fit()  # density fitting: faster
    mf.xc = xc
    mf.conv_tol = 1e-8
    mf.kernel()
    if not mf.converged:
        warnings.warn(f"SCF not converged (charge={charge})")
    return mf


def dft_descriptors(mol, site_atoms=None, conceptual=True, **kw):
    mf = run_dft(mol, **kw)
    homo, lumo = frontier_orbitals(mf.mo_energy * HARTREE2EV, mf.mo_occ)
    _, q_mull = mf.mulliken_pop(verbose=0)
    _, q_low = mf.mulliken_meta(verbose=0)      # meta-Löwdin: less basis-set dependent than Mulliken
    d = {
        "HOMO": homo, "LUMO": lumo, "gap": lumo - homo,
        "dipole": np.linalg.norm(mf.dip_moment(unit="Debye", verbose=0)),
    }
    if conceptual:
        cat, an = run_dft(mol, charge=+1, spin=1, **kw), run_dft(mol, charge=-1, spin=1, **kw)
        cdft = conceptual_dft(mf.e_tot * HARTREE2EV, cat.e_tot * HARTREE2EV, an.e_tot * HARTREE2EV,
                              q_low, cat.mulliken_meta(verbose=0)[1], an.mulliken_meta(verbose=0)[1])
        d.update({k: cdft[k] for k in ["IP", "EA", "chi", "eta", "omega"]})
    for label, idx in (site_atoms or {}).items():
        d[f"q_mulliken_{label}"] = q_mull[idx]
        d[f"q_lowdin_{label}"] = q_low[idx]
        if conceptual:
            d[f"f+_{label}"] = cdft["f_plus"][idx]
            d[f"f-_{label}"] = cdft["f_minus"][idx]
    return d, mf


# One molecule first, to see the timing
t0 = time.time()
d, mf_anisole = dft_descriptors(mols_xtb["4-bromoanisole"], aryl_halide_sites(mols_xtb["4-bromoanisole"]))
print(f"3 DFT calculations (N, N-1, N+1) took {time.time() - t0:.1f} s")
pd.Series(d).round(3)

> **Look at the electron affinities.** Most DFT EAs are *negative*. The anion is less stable than the neutral, and the extra electron would leave the molecule if it could. With a basis set lacking diffuse functions (def2-SVP), the electron is artificially squeezed onto the molecule. The N+1 quantities (EA, ω, f⁺) are therefore basis-set artefacts to a large extent. Compare f⁺ from xTB and DFT below, and see the exercises.

In [ ]:
# Whole series. Takes a few minutes on a laptop; results are cached to disk.
dft_cache = RESULTS / "hammett_dft_b3lyp_def2svp.csv"
if dft_cache.exists():
    desc_dft = pd.read_csv(dft_cache, index_col=0)
else:
    rows = {}
    for name, m in mols_xtb.items():
        t0 = time.time()
        rows[name], _ = dft_descriptors(m, aryl_halide_sites(m))
        print(f"{name:26s} {time.time() - t0:6.1f} s")
    desc_dft = pd.DataFrame(rows).T
    desc_dft.to_csv(dft_cache)
desc_dft.round(3)

### 5b. Looking at orbitals, density and electrostatic potential

Numbers are easier to trust once you've *seen* them. We write Gaussian cube files (3D grids) and display them with py3Dmol. You can also open `.cube` files in VMD, Avogadro, or VESTA.

- **HOMO / LUMO** isosurfaces: where are the frontier electrons? Is the LUMO on the C–Br σ\* bond or on the ring π\*?
- **Electrostatic potential (ESP) mapped onto the electron-density surface:** red = electron-rich (negative), blue = electron-poor (positive).

In [ ]:
def write_cubes(mf, stem, nx=60):
    homo = int(np.where(mf.mo_occ > 0)[0].max())
    dm = mf.make_rdm1()
    cubegen.orbital(mf.mol, f"{stem}_homo.cube", mf.mo_coeff[:, homo], nx=nx, ny=nx, nz=nx)
    cubegen.orbital(mf.mol, f"{stem}_lumo.cube", mf.mo_coeff[:, homo + 1], nx=nx, ny=nx, nz=nx)
    cubegen.density(mf.mol, f"{stem}_dens.cube", dm, nx=nx, ny=nx, nz=nx)
    cubegen.mep(mf.mol, f"{stem}_esp.cube", dm, nx=nx, ny=nx, nz=nx)


def show_orbital(mol, cube_file, iso=0.03, width=350, height=280):
    cube = open(cube_file).read()
    v = py3Dmol.view(width=width, height=height)
    v.addModel(Chem.MolToMolBlock(mol), "mol")
    v.setStyle({"stick": {}})
    v.addVolumetricData(cube, "cube", {"isoval": iso, "color": "blue", "opacity": 0.8})
    v.addVolumetricData(cube, "cube", {"isoval": -iso, "color": "red", "opacity": 0.8})
    v.zoomTo()
    return v.show()


def show_esp(mol, dens_file, esp_file, iso=0.002, vmin=-0.03, vmax=0.03, width=350, height=280):
    v = py3Dmol.view(width=width, height=height)
    v.addModel(Chem.MolToMolBlock(mol), "mol")
    v.setStyle({"stick": {}})
    v.addVolumetricData(open(dens_file).read(), "cube",
                        {"isoval": iso, "opacity": 0.9, "voldata": open(esp_file).read(),
                         "volformat": "cube", "volscheme": {"gradient": "rwb", "min": vmin, "max": vmax}})
    v.zoomTo()
    return v.show()


stem = str(RESULTS / "4-bromoanisole")
write_cubes(mf_anisole, stem)
m = mols_xtb["4-bromoanisole"]
show_orbital(m, f"{stem}_homo.cube")
show_orbital(m, f"{stem}_lumo.cube")
show_esp(m, f"{stem}_dens.cube", f"{stem}_esp.cube")

### 5c. Do the descriptors make chemical sense?

Plot each descriptor against σ<sub>p</sub>. A good electronic descriptor for the aryl halide should track the Hammett constant, and xTB and DFT should broadly agree.

In [ ]:
from scipy.stats import pearsonr

to_plot = ["HOMO", "LUMO", "IP", "EA", "omega", "f+_ipso"]
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, col in zip(axes.flat, to_plot):
    for desc, label, marker in [(desc_xtb, "GFN2-xTB", "o"), (desc_dft, "B3LYP/def2-SVP", "s")]:
        y = desc.loc[hammett.name, col].astype(float)
        r, _ = pearsonr(hammett.sigma_p, y)
        ax.scatter(hammett.sigma_p, y, marker=marker, label=f"{label} (r = {r:+.2f})")
    ax.set(xlabel="Hammett σp", ylabel=f"{col} (eV)" if col in ["HOMO", "LUMO", "IP", "EA", "omega"] else col, title=col)
    ax.legend(fontsize=8)
fig.tight_layout()

In [ ]:
# How well does cheap xTB reproduce DFT trends?
common = [c for c in desc_xtb.columns if c in desc_dft.columns]
comparison = pd.DataFrame({c: [pearsonr(desc_xtb[c].astype(float), desc_dft[c].astype(float))[0]] for c in common},
                          index=["r(xTB, DFT)"]).T
comparison.round(2)

### Your turn (Parts 2–5)

1. **Where is the LUMO?** For 4-bromonitrobenzene vs 4-bromoanisole, plot the LUMO. Is it on the C–Br σ\* or the ring π\*? What does that mean for using "LUMO energy" as a descriptor for oxidative addition?
2. **Charges are not observables.** Compare Mulliken, meta-Löwdin and Gasteiger charges on the ipso carbon across the series. Which ones correlate with σ<sub>p</sub>? Re-run one molecule with `basis="def2-tzvp"` and see how much each charge changes.
3. **Diffuse functions.** Recompute EA and f⁺ for two molecules with `basis="def2-svpd"` (or `aug-cc-pvdz`). How much do they change? Which N+1 descriptors would you trust?
4. **Koopmans vs ΔSCF.** Compare −ε<sub>HOMO</sub> with the ΔSCF IP from DFT. Why do they differ? (Hint: B3LYP has only 20% exact exchange.)
5. **Geometry sensitivity.** Compute DFT descriptors on the MMFF geometry instead of the xTB one. Which descriptors move most?
6. **Functional sensitivity.** Swap `xc="b3lyp"` for `xc="wb97x-d"` or `xc="pbe0"`. Do the *trends* (correlation with σ<sub>p</sub>) survive, even if absolute values shift?
7. **Solvation.** Real reactions are in solution. PySCF supports implicit solvation: `mf = mf.PCM(); mf.with_solvent.eps = 46.7` (DMSO). Does this change the descriptors meaningfully?

## 6. Export the descriptor table

Collect everything into one table, one row per aryl bromide, and save it. Notebook 03 reads this file to describe the substrates in the Bayesian optimisation.

If you computed descriptors for the boronic acids too, export them to a separate table in the same way (e.g. `results/boronic_acid_descriptors.csv`).

In [ ]:
export = (desc_rdkit.astype(float).add_prefix("rdkit_")
          .join(desc_xtb.astype(float).add_prefix("xtb_"))
          .join(desc_dft.astype(float).add_prefix("dft_")))
export.insert(0, "smiles", hammett.set_index("name").smiles)
export.insert(1, "substituent", hammett.set_index("name").substituent)
export.insert(2, "sigma_p", hammett.set_index("name").sigma_p)
export.index.name = "name"
export.to_csv(RESULTS / "aryl_bromide_descriptors.csv")
print(f"Saved {export.shape[0]} molecules × {export.shape[1]} columns to {RESULTS / 'aryl_bromide_descriptors.csv'}")
export.iloc[:, :8].round(3)

### Your turn 

1. Work out which descriptors are correlated by plotting correlation plots and working out correlation coefficient. 
2. Choose a **small** set of 3–5 descriptors that are not strongly correlated with each other but still cover the electronics of the series. Justify each choice chemically. This is the set you will use for learning yields. 
3. Compute the same descriptors for the **boronic acid partner(s)** you will use. Which atom-specific sites make sense there (e.g. B, the ipso carbon)?
4. Pd(0) oxidative addition into C–Br is the step the descriptors should "see". Would a descriptor computed on an oxidative-addition *transition state* or *product* be better than one computed on the free aryl bromide? What would it cost? (See Lu, Donnecke, Paci, Leitch, *Chem. Sci.* **2022**: a descriptor-based reactivity model for oxidative addition to Pd(0).)

---

## 7. Where next
- **Ready-made descriptor sets.** [Kraken](https://doi.org/10.1021/jacs.1c09718) (~1500 phosphine ligands with DFT descriptors, very relevant for choosing Suzuki ligands); [auto-QChem](https://github.com/doyle-lab-ucla/auto-qchem); ML-predicted QM descriptors ([qmdesc](https://github.com/yanfeiguan/qmdesc)).
- **Next notebook:** `02_suzuki_yield_prediction.ipynb`.